<a href="https://colab.research.google.com/drive/1uB-SEU89y16LklLLKS-H7JHbycFCiY75" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Music Generation with Transformer (GPT) using ABC Notation

Deep Learning Project

This notebook implements a **Transformer-based GPT model** to generate music using **ABC Notation**.

Features included:

- Dataset download from Kaggle
- Preprocessing and tokenization
- Data augmentation
- Train / Validation / Test split **(70 / 20 / 10)**
- PyTorch + PyTorch Lightning training
- Perplexity metric
- Attention visualization
- Temperature and Top-K sampling
- **Loss comparison graphs (Train vs Validation)**

## 1. Install Libraries

In [ ]:
# ============================================
# Se descarga y carga el dataset de ABC Notation.
# Se asegura disponibilidad local y se prepara para procesamiento.
# ============================================
%pip install transformers lightning pandas kaggle music21 matplotlib seaborn --quiet

In [ ]:
!apt-get install -y abcmidi timidity

In [ ]:
%pip install regex

## 2. Import Libraries

In [ ]:
import torch
import lightning as L
import matplotlib.pyplot as plt
import math
import random
import os
import re
import IPython.display as ipythondisplay

from torch.utils.data import Dataset, DataLoader, random_split
from transformers import GPT2Config, GPT2LMHeadModel

torch.manual_seed(42)

## 3. Download Dataset from Kaggle

In [ ]:
# ============================================
# En este bloque se descarga el dataset de ABC Notation desde Kaggle.
# Se valida si existe localmente para evitar descargas redundantes.
# ============================================
pip install kagglehub

In [ ]:
import os
import kagglehub

DATASET_DIR = "./dataset_abc"
DATA_FILE = os.path.join(DATASET_DIR, "dataabc.txt")

def download_dataset():
    if os.path.exists(DATA_FILE):
        print("Dataset already exists.")
        print("Path:", DATA_FILE)
        return DATASET_DIR
    print("Dataset not found. Downloading from Kaggle...")
    path = kagglehub.dataset_download(
        "juansebm/abc-notation-music-for-rnn"
    )
    print("Downloaded to:", path)
    return path


dataset_path = download_dataset()

In [ ]:
def extract_song_snippet(text):
    pattern = '(^|\n\n)(.*?)\n\n'
    search_results = re.findall(
        pattern,
        text,
        flags=re.DOTALL
    )
    songs = [song[1] for song in search_results]
    print("Found {} songs in text".format(len(songs)))
    return songs

In [ ]:

##Guardar cancion
def save_song_to_abc(song, filename="tmp"):
    save_name = "{}.abc".format(filename)
    with open(save_name, "w") as f:
        f.write(song)
    return filename

#Convertir ABC → WAV
def abc2wav(abc_file):
    suf = abc_file.rstrip('.abc')
    cmd = "abc2midi {} -o {}".format(
        abc_file,
        suf + ".mid"
    )
    os.system(cmd)
    cmd = "timidity {}.mid -Ow {}.wav".format(
        suf,
        suf
    )
    return os.system(cmd)

#Reproducir WAV
def play_wav(wav_file):
    return ipythondisplay.Audio(wav_file)

#Reproducir canción completa
def play_song(song):
    basename = save_song_to_abc(song)
    ret = abc2wav(basename + '.abc')
    if ret == 0:
        return play_wav(basename + '.wav')
    return None

## 4. Load Dataset

In [ ]:

# ============================================
# Carga y preprocesamiento del dataset ABC Notation
# Se limpian secuencias musicales y se preparan para tokenización
# ============================================
file_path = os.path.join(dataset_path, "dataabc.txt")
with open(file_path, "r", encoding="utf-8") as f:
    text = f.read()
print("Dataset length:", len(text))

In [ ]:
print(text[:1000])

In [ ]:
songs = extract_song_snippet(text)
example_song = songs[0]
print(example_song)


In [ ]:
play_song(example_song)

## 5. Preprocessing

In [ ]:
text = text.replace("\r","")
text = text.strip()

## 6. Character Tokenization

In [ ]:
# ============================================
# Conversión de texto musical (ABC Notation) a tokens numéricos.
# Permite al modelo Transformer procesar secuencias.
# ============================================
chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for ch,i in stoi.items()}

def encode(s):
    return [stoi[c] for c in s]

def decode(tokens):
    return "".join([itos[t] for t in tokens])

data = torch.tensor(encode(text), dtype=torch.long)
print("Vocabulary size:", vocab_size)

## 7. Data Augmentation

In [ ]:
# ============================================
# El data augmentation ayuda a que el modelo (tipo GPT-2) aprenda mejor patrones musicales sin necesitar más datos reales.
# Aumentar la diversidad del dataset sin recolectar más datos.
# Introduciendo ruido aleatorio en la secuencia para hacer el modelo más robusto.
# ============================================
def augment_sequence(seq, prob=0.01):
    seq = list(seq)
    for i in range(len(seq)):
        if random.random() < prob:
            seq[i] = random.choice(list(stoi.values()))
    return seq

## 8. Hyperparameters

In [ ]:
block_size = 128 # Longitud máxima de contexto que el modelo ve, La música ABC tiene patrones cortos → 128 es buen balance.

batch_size = 64 # Número de secuencias por iteración, 64 = equilibrio ideal.
learning_rate = 3e-4 # Qué tan rápido aprende el modelo, valor estándar en Transformers.
max_epochs = 10 # Número de veces que el modelo ve todo el dataset, 10 suficiente para aprender patrones básicos.

embedding_dim = 256 # Tamaño del vector que representa cada token,256 = balance perfecto.
num_layers = 6 # Número de capas Transformer, Dataset no es gigante → no necesitas más capas.
num_heads = 8 # Número de “perspectivas” de atención, cada head trabaja con 32 dimensiones.

## 9. Dataset Class

In [ ]:
class MusicDataset(Dataset):

    def __init__(self, data, block_size):
        self.data = data
        self.block_size = block_size

    def __len__(self):
        return len(self.data) - self.block_size

    def __getitem__(self, idx):
        x = self.data[idx:idx+block_size]
        y = self.data[idx+1:idx+block_size+1]
        return x, y

## 10. Train / Validation / Test Split (70 / 20 / 10)

In [ ]:
# ============================================
# División del dataset en:
# - 70% entrenamiento
# - 20% validación
# - 10% prueba
# ============================================
dataset = MusicDataset(data, block_size)

train_size = int(0.7 * len(dataset))
val_size = int(0.2 * len(dataset))
test_size = len(dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    dataset,
    [train_size, val_size, test_size]
)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

## 11. GPT Transformer Model

In [ ]:
# ============================================
# Definición del modelo autoregresivo basado en Transformers.
# Aprende patrones secuenciales en notación musical.
# ============================================
config = GPT2Config(

    vocab_size=vocab_size,
    n_positions=block_size,
    n_ctx=block_size,

    n_embd=embedding_dim,
    n_layer=num_layers,
    n_head=num_heads
)

model = GPT2LMHeadModel(config)

## 12. Lightning Model with Loss Tracking

In [ ]:
# ============================================
# Se crean DataLoaders para manejar batches durante entrenamiento
# y evaluación, optimizando el uso de memoria.
# ============================================
class GPTMusicModel(L.LightningModule):

    def __init__(self, model, lr=learning_rate):
        super().__init__()

        self.model = model
        self.lr = lr
        # loss final por epoch
        self.train_losses = []
        self.val_losses = []
        # acumuladores por batch
        self.train_step_losses = []
        self.val_step_losses = []


    def training_step(self, batch, batch_idx):
        x, y = batch
        outputs = self.model(input_ids=x, labels=y)
        loss = outputs.loss
        # guardar loss del batch
        self.train_step_losses.append(loss.item())
        self.log("train_loss", loss)
        return loss


    def validation_step(self, batch, batch_idx):
        x, y = batch
        outputs = self.model(input_ids=x, labels=y)
        loss = outputs.loss
        # guardar loss del batch
        self.val_step_losses.append(loss.item())
        self.log("val_loss", loss)


    def on_train_epoch_end(self):
        avg_loss = sum(self.train_step_losses) / len(self.train_step_losses)
        self.train_losses.append(avg_loss)
        self.train_step_losses.clear()


    def on_validation_epoch_end(self):
        if len(self.val_step_losses) == 0:
            return
        avg_loss = sum(self.val_step_losses) / len(self.val_step_losses)
        self.val_losses.append(avg_loss)
        self.val_step_losses.clear()


    def test_step(self, batch, batch_idx):

        x, y = batch
        outputs = self.model(input_ids=x, labels=y)
        loss = outputs.loss
        # importante: on_epoch=True para promedio automático
        self.log("test_loss", loss, on_epoch=True, prog_bar=True)
        return loss

    def configure_optimizers(self):
      return torch.optim.AdamW(self.parameters(), lr=self.lr)

## 13. Training

In [ ]:
# ============================================
# Entrenamiento del modelo usando PyTorch Lightning.
# Se optimiza la función de pérdida CrossEntropy.
# ============================================
model_module = GPTMusicModel(model)

trainer = L.Trainer(
    max_epochs=max_epochs,
    accelerator="auto",
    devices=1
)

trainer.fit(model_module, train_loader, val_loader)

## 14. Loss Comparison Graph (Train vs Validation)

In [ ]:
import matplotlib.pyplot as plt

# ============================================
# Se grafican pérdidas de entrenamiento y validación por época
# para evaluar convergencia del modelo.
# ============================================
epochs = range(len(model_module.train_losses))

plt.figure()

plt.plot(epochs, model_module.train_losses, label="Train Loss")
plt.plot(epochs, model_module.val_losses[:len(model_module.train_losses)], label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Train vs Validation Loss per Epoch")

plt.legend()
plt.show()

In [ ]:
import numpy as np

train_perplexity = np.exp(model_module.train_losses)
val_perplexity = np.exp(model_module.val_losses)

## 15. Test Evaluation

In [ ]:
# ============================================
# Evaluación del modelo en el conjunto de test.
# Se calcula:
# - Test Loss
# - Perplexity (exp(loss))
# ============================================
results = trainer.test(model_module, dataloaders=test_loader)
test_loss = results[0]["test_loss"]

print(f"Test Loss: {test_loss:.4f}")


## 16. Perplexity Metric

In [ ]:
# ============================================
# qué tan “confundido” está el modelo al predecir el siguiente token.
# Perplexity baja → repite patrones simples → mala música.
# Perplexity un poco mayor → más creatividad → mejor música.
# ============================================
perplexity = math.exp(test_loss)
print(f"Perplexity: {perplexity:.4f}")

## 17. Attention Visualization

In [ ]:
x,y = next(iter(test_loader))
outputs = model_module.model(
    input_ids=x,
    labels=y,
    output_attentions=True
)
attention = outputs.attentions[0][0,0].detach().cpu()
plt.imshow(attention)
plt.title("Attention Map")
plt.colorbar()
plt.show()

## 18. Music Generation with Temperature and Top-K Sampling

In [ ]:
# ============================================
# Generación autoregresiva de secuencias musicales.
# Se utiliza:
# - Temperature (control de creatividad)
# - Top-K Sampling (reducción de ruido)
# ============================================
def generate_music(model, start_text, length=400, temperature=0.8, top_k=20):
    device = next(model.parameters()).device
    model.eval()
    tokens = torch.tensor(encode(start_text)).unsqueeze(0).to(device)
    with torch.inference_mode():
        for _ in range(length):
            tokens_cond = tokens[:, -block_size:]
            outputs = model(input_ids=tokens_cond)
            logits = outputs.logits[:, -1, :] / temperature
            if top_k is not None:
                values, indices = torch.topk(logits, top_k)
                logits_filtered = torch.full_like(logits, float('-inf'))
                logits_filtered.scatter_(1, indices, values)
                logits = logits_filtered

            probs = torch.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            tokens = torch.cat([tokens, next_token], dim=1)
    return decode(tokens[0].tolist())

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)

## 19. Generate Music

In [ ]:
# ============================================
# Conversión de ABC Notation a MIDI y WAV.
# Permite escuchar las melodías generadas.
# seed = tonalidad (K:C),ritmo (M:4/4)
# ============================================
seed = "X:1\nT:Generated\nM:4/4\nK:C\n"
music = generate_music(model_module.model, seed, temperature=0.9)
print(music)

In [ ]:
play_song(music)

In [ ]:
seed = "X:1\nT:Generated\nM:4/4\nK:C\n"

generated_music = generate_music(
    model_module.model,
    seed,
    temperature=0.8
)
print(generated_music)

In [ ]:
play_song(generated_music)

## 20. Full Pipeline

ABC Dataset (Kaggle)  
↓  
Dataset Download  
↓  
Preprocessing  
↓  
Tokenization  
↓  
Data Augmentation  
↓  
Train / Validation / Test Split (70 / 20 / 10)  
↓  
PyTorch Dataset + DataLoader  
↓  
Transformer GPT Model  
↓  
Training (PyTorch Lightning)  
↓  
Loss Monitoring (Train vs Validation)  
↓  
Evaluation (Test + Perplexity)  
↓  
Attention Visualization  
↓  
Music Generation (Temperature + Top-K Sampling)  
↓  
ABC Output